# 10 · Checkpoint test evaluation

Restores the best checkpoint of each of the 36 benchmark trials (6 architectures × 2 datasets × 3 optimizers)
and scores the network's own sigmoid output on the **same** test split used in training
(70/15/15 stratified, seed 42, train-only z-scoring). Nothing is retrained.

Outputs, per trial: exact (rank-based) test AUC and test AUPRC (average precision).
Result file: `network_test_results_36_trials.csv` in `OUTPUT_DIR` (copy it to `Results/`).

**Requirements**
- The checkpoints written by notebooks 03 to 08 in `OUTPUT_DIR/checkpoints/` (`<model>_<A|B>_<optimizer>_best.keras`).
- `dataset_A.csv` and `dataset_B.csv` in `OUTPUT_DIR` (from notebook 01).
- A GPU runtime is recommended (inference only, about 5 to 10 minutes per architecture).

The model definitions are not copied here. The notebook runs the definition cells of notebooks 03 to 08
from this repository and skips their training loop, so the architectures are exactly the ones that were trained.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/epilepsy_outputs'      # same folder as in notebooks 01-08
REPO_URL   = 'https://github.com/ziadmohamdd/EEG-Seizure-Detection-CHB-MIT-Multi-Model-Pipeline'
REPO_DIR   = '/content/repo'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
PHASES = os.path.join(REPO_DIR, 'Phases')
print(sorted(os.listdir(PHASES)))


Mounted at /content/drive
['01_Phase1_Preprocessing.ipynb', '02_Utils_ModelUtils.ipynb', '03_Model1_CNN2D.ipynb', '03b_CNN2D_ECG_Experiment.ipynb', '04_Model2_Conformer.ipynb', '05_Model3_LSTM.ipynb', '06_Model4_CNN_LSTM.ipynb', '07_Model5_LSTM_Conformer.ipynb', '08_Model6_ComplexHybrid.ipynb', '09_Phase3_ResultsCompilation.ipynb']


In [2]:
import json
from IPython import get_ipython

def load_definitions(nb_file):
    """Run every code cell of a model notebook except its training loop."""
    nb = json.load(open(os.path.join(PHASES, nb_file)))
    ip = get_ipython()
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        src = ''.join(cell['source'])
        if 'for ds_label' in src:          # the training loop: skip it
            continue
        res = ip.run_cell(src, silent=True)
        if res.error_in_exec is not None:
            raise RuntimeError(f'{nb_file}: a definition cell failed') from res.error_in_exec


In [3]:
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# notebook, checkpoint prefix, build function, tensor function, sequence input?
MODELS = [
    ('03_Model1_CNN2D.ipynb',          'CNN2D',          'build_cnn2d',          'prepare_tensors_2d',  False),
    ('04_Model2_Conformer.ipynb',      'conformer',      'build_conformer',      'prepare_tensors_seq', True),
    ('05_Model3_LSTM.ipynb',           'lstm',           'build_lstm',           'prepare_tensors_seq', True),
    ('06_Model4_CNN_LSTM.ipynb',       'cnn_lstm',       'build_cnn_lstm',       'prepare_tensors_seq', True),
    ('07_Model5_LSTM_Conformer.ipynb', 'lstm_conformer', 'build_lstm_conformer', 'prepare_tensors_seq', True),
    ('08_Model6_ComplexHybrid.ipynb',  'complex_hybrid', 'build_complex_hybrid', 'prepare_tensors_seq', True),
]
OPTIMIZERS = ['Adam', 'Nadam', 'AdamW']
DATASETS = [('A', os.path.join(OUTPUT_DIR, 'dataset_A.csv')), ('B', os.path.join(OUTPUT_DIR, 'dataset_B.csv'))]

def evaluate_model(nb_file, prefix, build_name, prep_name, seq):
    load_definitions(nb_file)
    g = globals()
    build, prep = g[build_name], g[prep_name]
    rows = []
    for ds_label, ds_path in DATASETS:
        X, y = prep(load_dataset(ds_path))
        # identical split and scaling to the training notebooks
        Xtrv, Xte, ytrv, yte = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEED)
        Xtr, Xv, ytr, yv = train_test_split(Xtrv, ytrv, test_size=0.15/0.85, stratify=ytrv, random_state=SEED)
        Xtr, Xv, Xte = normalize_tensors(Xtr, Xv, Xte)
        for opt in OPTIMIZERS:
            ckpt = os.path.join(OUTPUT_DIR, 'checkpoints', f'{prefix}_{ds_label}_{opt}_best.keras')
            if not os.path.exists(ckpt):
                print('MISSING', ckpt); continue
            if seq:
                m, _ = build(n_bands=X.shape[1], n_channels=X.shape[2])
            else:
                m, _ = build(n_channels=X.shape[1], n_bands=X.shape[2])
            m.load_weights(ckpt)
            p = m.predict(Xte, batch_size=2048, verbose=0).ravel()
            r = dict(model=prefix.lower(), dataset=ds_label, optimizer=opt,
                     n_test=len(yte), n_test_pos=int(yte.sum()),
                     test_auc=round(float(roc_auc_score(yte, p)), 4),
                     test_auprc=round(float(average_precision_score(yte, p)), 4))
            print(r); rows.append(r)
            del m; tf.keras.backend.clear_session()
    return rows


In [4]:
all_rows = []
for spec in MODELS:
    print('\n==========', spec[0])
    all_rows += evaluate_model(*spec)

res = pd.DataFrame(all_rows)
out = os.path.join(OUTPUT_DIR, 'network_test_results_36_trials.csv')
res.to_csv(out, index=False)
print('\nSaved', out, '-', len(res), 'trials')
res



========== 03_Model1_CNN2D.ipynb
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sat Nov 14 20:39:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB | 

Model: "CNN2D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ eeg_input (InputLayer)          │ (None, 18, 5, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 18, 5, 32)      │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 18, 5, 32)      │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 9, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d               │ (None, 9, 5, 32)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 9, 5, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 9, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_1             │ (None, 5, 5, 64)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 5, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 5, 5, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_2             │ (None, 5, 5, 128)      │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,577 (189.75 KB)

 Trainable params: 48,129 (188.00 KB)

 Non-trainable params: 448 (1.75 KB)

... (28 outputs left out)

 Trainable params: 405,697 (1.55 MB)

 Non-trainable params: 832 (3.25 KB)

Feature extractor output: (None, 256)
    [nan guard] dropping 7 rows with NaN features
{'model': 'complex_hybrid', 'dataset': 'A', 'optimizer': 'Adam', 'n_test': 103677, 'n_test_pos': 210, 'test_auc': 0.872, 'test_auprc': 0.0214}
{'model': 'complex_hybrid', 'dataset': 'A', 'optimizer': 'Nadam', 'n_test': 103677, 'n_test_pos': 210, 'test_auc': 0.8286, 'test_auprc': 0.0157}
{'model': 'complex_hybrid', 'dataset': 'A', 'optimizer': 'AdamW', 'n_test': 103677, 'n_test_pos': 210, 'test_auc': 0.7704, 'test_auprc': 0.0047}
{'model': 'complex_hybrid', 'dataset': 'B', 'optimizer': 'Adam', 'n_test': 24658, 'n_test_pos': 16, 'test_auc': 0.936, 'test_auprc': 0.0123}
{'model': 'complex_hybrid', 'dataset': 'B', 'optimizer': 'Nadam', 'n_test': 24658, 'n_test_pos': 16, 'test_auc': 0.9781, 'test_auprc': 0.0319}
{'model': 'complex_hybrid', 'dataset': 'B', 'optimizer': 'AdamW', 'n_test': 24658, 'n_test_pos': 16, 'test_auc': 0.94, 'test_auprc': 0.2165}

Saved /content/drive/MyDrive/epilepsy_outputs/network

,model,dataset,optimizer,n_test,n_test_pos,test_auc,test_auprc
0,cnn2d,A,Adam,103677,210,0.8624,0.0398
1,cnn2d,A,Nadam,103677,210,0.7983,0.0105
2,cnn2d,A,AdamW,103677,210,0.8752,0.0373
3,cnn2d,B,Adam,24658,16,0.9094,0.1145
4,cnn2d,B,Nadam,24658,16,0.7639,0.0866
5,cnn2d,B,AdamW,24658,16,0.6677,0.0845
6,conformer,A,Adam,103677,210,0.8758,0.0207
7,conformer,A,Nadam,103677,210,0.8872,0.0375
8,conformer,A,AdamW,103677,210,0.8230,0.0069
9,conformer,B,Adam,24658,16,0.8214,0.0437


## Optional: compare with validation AUC

Joins the new results with `results.csv` from the training notebooks and reports the Spearman
correlation between validation AUC (Keras, binned) and the exact test AUC.


In [5]:
from scipy import stats
val = pd.read_csv(os.path.join(REPO_DIR, 'Results', 'results.csv'))
val['model'] = val['model'].str.lower(); val['dataset'] = val['dataset'].str[-1]
j = val.merge(res, on=['model', 'dataset', 'optimizer'])
print(len(j), 'trials joined')
print('Spearman val AUC vs test AUC:', stats.spearmanr(j.best_val_auc, j.test_auc))
j[['model', 'dataset', 'optimizer', 'best_val_auc', 'test_auc', 'test_auprc']]


36 trials joined
Spearman val AUC vs test AUC: SignificanceResult(statistic=np.float64(0.8004119348538503), pvalue=np.float64(4.642680988499059e-09))


,model,dataset,optimizer,best_val_auc,test_auc,test_auprc
0,cnn2d,A,Adam,0.8489,0.8624,0.0398
1,cnn2d,A,Nadam,0.8211,0.7983,0.0105
2,cnn2d,A,AdamW,0.8760,0.8752,0.0373
3,cnn2d,B,Adam,0.9714,0.9094,0.1145
4,cnn2d,B,Nadam,0.8131,0.7639,0.0866
5,cnn2d,B,AdamW,0.8825,0.6677,0.0845
6,conformer,A,Adam,0.8449,0.8758,0.0207
7,conformer,A,Nadam,0.7963,0.8872,0.0375
8,conformer,A,AdamW,0.5003,0.8230,0.0069
9,conformer,B,Adam,0.8146,0.8214,0.0437
